In [0]:
%sql
-- F1 Silver Layer – SCD4 History Tables
-- Run once to create history tables for each FACT table.
-- These mirror the FACT table schema plus an `archived_at` audit column.

-- ── results_history ──────────────────────────────────────────
CREATE TABLE IF NOT EXISTS F1.Silver.results_history
(
    result_id         INT,
    race_id           INT,
    driver_id         INT,
    constructor_id    INT,
    number            INT,
    grid              INT,
    position          STRING,
    position_text     INT,
    position_order    INT,
    points            INT,
    laps              INT,
    time              STRING,
    milliseconds      INT,
    fastest_lap       INT,
    rank              INT,
    fastest_lap_time  STRING,
    fastest_lap_speed FLOAT,
    status_id         STRING,
    ingestion_date    TIMESTAMP,
    data_source       STRING,
    file_date         STRING,
    row_hash          STRING,
    eff_start_date    DATE,
    eff_end_date      DATE,
    is_current        INT,
    created_at        TIMESTAMP,
    updated_at        TIMESTAMP,
    archived_at       TIMESTAMP   -- SCD4 audit column
)
USING DELTA
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/F1.Silver/results_history';

-- ── lap_times_history ────────────────────────────────────────
CREATE TABLE IF NOT EXISTS F1.Silver.lap_times_history
(
    race_id        INT,
    driver_id      INT,
    lap            INT,
    position       INT,
    time           STRING,
    milliseconds   INT,
    ingestion_date TIMESTAMP,
    data_source    STRING,
    file_date      STRING,
    row_hash       STRING,
    eff_start_date DATE,
    eff_end_date   DATE,
    is_current     INT,
    created_at     TIMESTAMP,
    updated_at     TIMESTAMP,
    archived_at    TIMESTAMP
)
USING DELTA
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/F1.Silver/lap_times_history';

-- ── pit_stops_history ────────────────────────────────────────
CREATE TABLE IF NOT EXISTS F1.Silver.pit_stops_history
(
    driver_id      INT,
    race_id        INT,
    stop           INT,
    lap            INT,
    time           STRING,
    duration       STRING,
    milliseconds   INT,
    ingestion_date TIMESTAMP,
    data_source    STRING,
    file_date      STRING,
    row_hash       STRING,
    eff_start_date DATE,
    eff_end_date   DATE,
    is_current     INT,
    created_at     TIMESTAMP,
    updated_at     TIMESTAMP,
    archived_at    TIMESTAMP
)
USING DELTA
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/F1.Silver/pit_stops_history';

-- ── qualifying_history ───────────────────────────────────────
CREATE TABLE IF NOT EXISTS F1.Silver.qualifying_history
(
    qualify_id     INT,
    race_id        INT,
    driver_id      INT,
    constructor_id INT,
    number         INT,
    position       INT,
    q1             STRING,
    q2             STRING,
    q3             STRING,
    ingestion_date TIMESTAMP,
    data_source    STRING,
    file_date      STRING,
    row_hash       STRING,
    eff_start_date DATE,
    eff_end_date   DATE,
    is_current     INT,
    created_at     TIMESTAMP,
    updated_at     TIMESTAMP,
    archived_at    TIMESTAMP
)
USING DELTA
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/F1.Silver/qualifying_history';


In [0]:
%sql
sELECT * FROM F1.silver.etl_batch_status;

In [0]:
from pyspark.sql import functions as F
cfg = (
    spark.table("f1.silver.etl_batch_status")
    .filter((F.col("table_name") == "circuits") & (F.col("status") == "SUCCESS"))
    .orderBy(F.col("batch_date").desc())
    .limit(1)
    )
row = cfg.collect()
display(row)